# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore a dataset described by a Croissant schema using the `mlcroissant` library.

### Dataset Source
The dataset schema is accessible via the following Croissant schema URL:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and available record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata (not as a dict)
metadata = dataset.metadata
print(f"Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'Unknown')}")

## 2. Data Overview
Let us list all record sets, fields, and key `@id`s available in this dataset schema.

In [ ]:
# List record sets with their @id and name
print("--- Record Sets ---")
record_sets = dataset.record_sets()
if not record_sets:
    print("No record sets explicitly listed, inferring from files...")
    # In some datasets, record sets may be identified as distributions/files.
    files = getattr(metadata, 'distribution', [])
    for file_obj in files:
        # Each file/distribution has an @id and may serve as a record set
        print(f"FileObject @id: {file_obj.get('@id')}")
        # Try to access file name if present
        print(f"  Encoding: {file_obj.get('encodingFormat', 'unknown')}")
        print(f"  URL: {file_obj.get('contentUrl', 'unknown')}")
else:
    for rs in record_sets:
        print(f"@id: {rs['@id']}, name: {rs.get('name', 'no name available')}")
        # List fields for each record set
        if 'field' in rs:
            print("    Fields:")
            for field in rs['field']:
                print(f"      - @id: {field['@id']}, name: {field.get('name', 'no name')}")

# If mlcroissant provides dataset.fields(), you could also enumerate all fields
print("\n--- Fields in the dataset ---")
fields = dataset.fields()
if fields:
    for field in fields:
        print(f"@id: {field['@id']}, name: {field.get('name', 'no name')}, type: {field.get('dataType', 'unknown')}")

## 3. Data Extraction
Load one or more record sets into pandas DataFrames. We will use the record set and field `@id`s from the overview above.

**Note:** This dataset contains one main data file (record set):

`@id`: `https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034`

Let us extract records with `mlcroissant` using this `@id`.

In [ ]:
# Choose the record set (data file) @id from the overview
record_sets = ["https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034"]
dataframes = {}

for record_set_id in record_sets:
    # Load records as list of dicts (each record is a row)
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for {record_set_id}\n")
    else:
        print(f"No records found for {record_set_id}")

# Show columns available in the main record set
main_record_set_id = record_sets[0]
if main_record_set_id in dataframes:
    print("Columns:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No dataframe available for the main record set.")

## 4. Exploratory Data Analysis (EDA)
Now let's analyze the protein data from the main record set. We'll pick a numeric field and a group/categorical field, using their `@id` values as the *column names*:

- Example numeric fields: `cr:field-coverage`, `cr:field-peptide_count`, `cr:field-MW`
- Example group fields: `cr:field-description`, `cr:field-accession`

> **TIP:** The actual column names depend on how the `mlcroissant` library renders fields, usually mapped from their `@id`.

We'll assume `cr:field-peptide_count` is a valid numeric field, and `cr:field-description` is a group field.

In [ ]:
main_record_set_id = "https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034"
df = dataframes[main_record_set_id]

# Set field IDs (these should match your DataFrame's columns)
numeric_field_id = "cr:field-peptide_count"
group_field_id = "cr:field-description"

# If the expected ID differs, print columns to choose available ones
if numeric_field_id not in df.columns:
    print("Available columns:", df.columns.tolist())

# Filter records where peptide_count > threshold
threshold = 10
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} (z-score):")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by description and get mean peptide_count
    if group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean peptide_count by description:")
        display(grouped_df.head())
else:
    print(f"Numeric field {numeric_field_id} not found in columns. Please review field names above.")

## 5. Visualization
Let's visualize the distribution of peptide counts and the top 10 most abundant protein descriptions (by summed peptide counts).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

numeric_field = numeric_field_id
group_field = group_field_id

if numeric_field in df.columns and group_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], bins=30, kde=True)
    plt.title('Distribution of Peptide Counts')
    plt.xlabel('Peptide Count')
    plt.ylabel('Number of Proteins')
    plt.show()

    top_descriptions = (
        df.groupby(group_field)[numeric_field].sum().sort_values(ascending=False).head(10)
    )
    plt.figure(figsize=(10,5))
    sns.barplot(y=top_descriptions.index, x=top_descriptions.values, orient='h')
    plt.title('Top 10 Protein Descriptions by Total Peptide Count')
    plt.xlabel('Total Peptide Count')
    plt.ylabel('Description')
    plt.tight_layout()
    plt.show()
else:
    print("Required columns for visualization not found. Please adjust field IDs or review earlier output.")

## 6. Conclusion
We have loaded, examined, and visualized the protein abundance data from the FAIR² dataset using `mlcroissant` directly via its Croissant schema.

**Key steps performed:**
- Loaded dataset schema and metadata.
- Identified the main record set and reviewed field `@id`s.
- Loaded protein abundance tabular data and displayed available columns.
- Filtered, normalized, and grouped peptide count data using field IDs.
- Visualized the distribution of peptide counts and the top protein descriptions by abundance.

You can further explore other numeric or categorical fields using their `@id` as the column name in the DataFrame. For more advanced analysis, see the [mlcroissant documentation](https://mlcommons.github.io/croissant/api/mlcroissant/).
